# 04 — SAMURAI: motion-aware memory for long videos

**This notebook depends on nothing else.** It is training-free, changes no
weights, and needs no new engine. Run it first if you like.

## The failure it targets

EdgeTAM emits an `object_score_logits` per frame. When it drops below zero the
mask is zeroed **and a constant "no object" vector (`no_obj_ptr`) is written
into the memory bank** instead of a real appearance. The next seven frames read
that back.

The zeroed mask is not the problem — that frame was bad anyway. The memory is.
**One hard frame poisons it, and nothing in the architecture ever un-poisons
it.** That is a policy problem, not a precision one, so no amount of fp16, INT8
or fine-tuning touches it.

## What changes

| | stock SAM 2 | SAMURAI |
|---|---|---|
| which of 3 candidate masks | highest IoU-head score | `0.15·s_kf + 0.85·s_mask`, where `s_kf` is agreement with a Kalman prediction |
| which frames enter memory | the last N, unconditionally | only if `iou > 0.5` **and** `object_score > 0` **and** `kf_score > 0` |

That middle threshold is exactly where EdgeTAM starts writing `no_obj_ptr`.

## Why it costs about nothing

The memory-bank bookkeeping is the part this project **deliberately left in
PyTorch** (`docs/tensorrt_fp16.md`, *what stays in PyTorch*). That is precisely
what SAMURAI modifies — so no engine changes shape, and the Kalman filter is 8
states on numpy. On the TensorRT path the only measurable cost appears when
SAMURAI's choice differs from the engine's: one 128²→512² bilinear upsample,
~0.2 ms.

In [ ]:
import os, sys
from pathlib import Path

REPO = Path("/content/sam-dedection")
if not REPO.exists():
    !git clone -q https://github.com/yigitkayabagci/sam-dedection.git {REPO}
os.chdir(REPO); sys.path.insert(0, str(REPO))

!bash scripts/setup_edgetam.sh 2>&1 | tail -3
!pip install -q -r requirements.txt

# 39 cases covering the Kalman filter, the selection rule and the memory gate.
# None of them need EdgeTAM, a GPU or a checkpoint.
!python -m unittest tests.test_samurai 2>&1 | tail -3

In [ ]:
# --- Data on local disk, keepsakes on Drive -----------------------------
# The dataset is a few hundred thousand small JPEGs; the Drive FUSE mount
# serves those an order of magnitude slower than the GPU reads them. Drive
# holds only what is worth surviving the runtime: the checkpoint and the RLE
# label store, both megabytes.
DATA_DIR = Path("/content/data")
!python tools/fetch_antiuav410.py --dest {DATA_DIR} --splits train val

from tools.fetch_antiuav410 import dataset_root, describe, find_splits

splits = find_splits(DATA_DIR)
DATA = dataset_root(splits)
print(describe(splits))

WORK = Path("/content/work")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/edgetam-thermal")
except Exception as exc:
    print(f"no Drive ({type(exc).__name__}) -- WORK stays at {WORK}")
WORK.mkdir(parents=True, exist_ok=True)

import shutil
CKPT = REPO / "checkpoints"; CKPT.mkdir(exist_ok=True)
# Whichever adapted checkpoint is on Drive, LoRA first: it is the 512 thermal
# default (docs/lora_vs_finetune.md). Stock works too -- nothing here touches
# weights -- it is just a weaker starting point.
CONFIG = "configs/edgetam_512.yaml"
for name, config in (("edgetam_lora_512.pt", "configs/edgetam_512_lora.yaml"),
                     ("edgetam_thermal_512.pt", "configs/edgetam_512_thermal.yaml")):
    if (WORK / name).exists():
        shutil.copy(WORK / name, CKPT / name)
        CONFIG = config
        break
print(f"baseline config: {CONFIG}")

## First: does the failure actually happen here?

Before measuring a fix, confirm the thing being fixed. This tracks validation
sequences with stock EdgeTAM and reports **dropout episodes** — runs of frames
where a visible target was not tracked.

The distinction that matters, and the one a mean IoU cannot make: *one
60-frame loss* and *sixty 1-frame losses* score identically and are completely
different bugs. Memory poisoning is firmly the first kind. If the episodes here
are short and scattered, this notebook is solving the wrong problem for your
footage — and that is worth knowing before spending an hour on it.

In [ ]:
!python tools/eval_antiuav.py --data {DATA} --split val --limit 20 --mode crop \
    --tracker edgetam --config {CONFIG} --json results/stock.json 2>&1 | tail -25

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

stock = json.loads(Path("results/stock.json").read_text())["sequences"]
lengths = [n for s in stock for n in s["dropout_lengths"]]
if not lengths:
    print("No dropouts on this subset -- nothing here to recover. Raise --limit, "
          "or pick sequences with the Occlusion / Out-of-View attributes.")
else:
    print(f"{len(lengths)} episode(s), {sum(lengths)} frames lost, "
          f"median {np.median(lengths):.0f}, longest {max(lengths)}")
    plt.figure(figsize=(6, 3.2))
    plt.hist(lengths, bins=range(1, max(lengths) + 2), color="#eb6834")
    plt.xlabel("dropout length (frames)"); plt.ylabel("episodes")
    plt.title("stock EdgeTAM: how long a lost target stays lost")
    plt.tight_layout(); plt.show()
    print("\nA long right tail is the memory-poisoning signature: the tracker "
          "does not recover on its own, it waits for the object to reappear "
          "somewhere the poisoned memory happens to accept.")

## The fix, at its published defaults

In [ ]:
!python tools/eval_antiuav.py --data {DATA} --split val --limit 20 --mode crop \
    --tracker edgetam --config configs/edgetam_samurai_512.yaml \
    --json results/samurai.json 2>&1 | tail -25

In [ ]:
def summarise(rows):
    frames = sum(r["frames"] for r in rows)
    lengths = [n for r in rows for n in r["dropout_lengths"]]
    return {
        "state accuracy": sum(r["state_accuracy"] * r["frames"] for r in rows) / frames,
        "success AUC": sum(r["success_auc"] * r["frames"] for r in rows) / frames,
        "episodes": len(lengths),
        "frames lost": sum(lengths),
        "longest": max(lengths, default=0),
    }

samurai = json.loads(Path("results/samurai.json").read_text())["sequences"]
before, after = summarise(stock), summarise(samurai)

print(f"{'':<16}{'stock':>12}{'samurai':>12}{'change':>12}")
for key in before:
    delta = after[key] - before[key]
    fmt = "{:>12.4f}" if isinstance(before[key], float) else "{:>12}"
    print(f"{key:<16}" + fmt.format(before[key]) + fmt.format(after[key])
          + (f"{delta:>+12.4f}" if isinstance(delta, float) else f"{delta:>+12}"))

print("\nRead the episode columns, not just the accuracy:")
print("  episodes down, lengths down  -> the memory gate is working")
print("  episodes down, lengths same  -> only the mask re-selection helped;")
print("                                  the gate is too loose. Raise memory_iou.")
print("  episodes same, lengths down  -> recovery improved, which is the point")

## Sweeping the gate

Two knobs decide almost everything. `memory_obj_score` is the direct one — it
is the threshold at which EdgeTAM starts writing `no_obj_ptr` — and
`memory_iou` guards against remembering a frame that was tracked confidently
but wrongly.

Too strict and the bank starves: with nothing acceptable to remember, the
tracker runs on the conditioning frame alone. Too loose and it is stock
behaviour. The sweep below shows where that boundary sits on your data rather
than on the paper's.

In [ ]:
import itertools, yaml
from tqdm.auto import tqdm

base = yaml.safe_load(Path("configs/edgetam_samurai_512.yaml").read_text())
grid = list(itertools.product([0.3, 0.5, 0.7], [0.0, 0.5, 1.0]))
sweep = []

for memory_iou, obj_score in tqdm(grid, desc="sweep"):
    cfg = {**base, "samurai": {**base["samurai"], "memory_iou": memory_iou,
                               "memory_obj_score": obj_score}}
    Path("configs/_sweep.yaml").write_text(yaml.safe_dump(cfg))
    out = Path(f"results/sweep_{memory_iou}_{obj_score}.json")
    !python tools/eval_antiuav.py --data {DATA} --split val --limit 8 --mode crop \
        --tracker edgetam --config configs/_sweep.yaml --json {out} > /dev/null 2>&1
    if out.exists():
        rows = json.loads(out.read_text())["sequences"]
        sweep.append({"memory_iou": memory_iou, "obj_score": obj_score, **summarise(rows)})

print(f"{'mem_iou':>8}{'obj':>6}{'state acc':>12}{'episodes':>10}{'longest':>9}")
for row in sweep:
    print(f"{row['memory_iou']:>8}{row['obj_score']:>6}"
          f"{row['state accuracy']:>12.4f}{row['episodes']:>10}{row['longest']:>9}")
Path("configs/_sweep.yaml").unlink(missing_ok=True)

In [ ]:
# --- Watch one recovery ------------------------------------------------
# The number that matters is a length; this is what that length looks like.
worst = max(stock, key=lambda s: max(s["dropout_lengths"], default=0))
print(f"longest stock dropout: {max(worst['dropout_lengths'])} frames "
      f"in {worst['name']}")

for label, config in (("stock", CONFIG), ("samurai", "configs/edgetam_samurai_512.yaml")):
    !python cli.py --tracker edgetam --config {config} \
        --frames-dir {DATA}/val/{worst['name']} --frame-pattern '*.jpg' \
        --prompt file --prompt-file {DATA}/val/{worst['name']}/prompts.json \
        --output outputs/{worst['name']}_{label}.mp4 --fps 30 2>&1 | tail -3
print("\nPlay the two side by side. The stock run loses the target and stays "
      "lost; if the gate is doing its job the SAMURAI run re-acquires.")

## On the TensorRT path

The `sam_head` engine picks its candidate internally, so the mask re-selection
needs one extra engine output: **every candidate's object pointer**, not just
the winner's. The pointer written to the memory bank has to describe the mask
that was kept.

```bash
python tools/export_edgetam_onnx.py --outdir models512/ --image-size 512 \
    --checkpoint checkpoints/edgetam_lora_512.pt --all-pointers --verify
python tools/build_trt_engines.py --outdir models512/ --max-batch 4
```

Then add the same `samurai:` block to `configs/edgetam_trt_512.yaml`.

**Existing engines keep working.** Without `obj_ptr_all` the tracker says so
once and disables the mask re-selection *only* — the memory gate still applies,
because it never left PyTorch. That is most of the benefit for no rebuild.

## What to check before believing any of it

`tests/test_edgetam_trt_integration.py::test_samurai_with_a_permissive_config_is_stock_edgetam`
asserts that thresholds rejecting nothing, with `kf_weight: 0`, reproduce stock
EdgeTAM **pixel for pixel**. Without that guarantee, "SAMURAI helped" and
"SAMURAI changed something unrelated" are indistinguishable.

```bash
python -m pytest tests/test_edgetam_trt_integration.py -k samurai -v
```

## Why not SAM2Long

Same problem, also training-free, better published long-video numbers — and it
keeps N parallel memory pathways. Memory attention is **61.9 % of the per-frame
arithmetic**, so three pathways puts a 10 ms frame near 22 ms: inside the
20–30 ms ceiling, but spending the entire margin on one technique. SAMURAI
costs approximately nothing. If it falls short on the Occlusion / Out-of-View
slices, SAM2Long is next and the budget is there.

**Next:** `05_adaptive_inference_sahi.ipynb` — where the rest of that margin
goes.